In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Data Exploration Notebook\n",
    "## Systematic Review Data Analysis\n",
    "\n",
    "This notebook explores RIS and Covidence data for systematic reviews."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import rispy\n",
    "import json\n",
    "import yaml\n",
    "from pathlib import Path\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set style\n",
    "plt.style.use('seaborn-v0_8-whitegrid')\n",
    "sns.set_palette(\"Set2\")\n",
    "\n",
    "# Display options\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_rows', 100)\n",
    "pd.set_option('display.max_colwidth', 200)\n",
    "\n",
    "print(\"✅ Libraries imported\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Configuration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load configuration\n",
    "config_path = Path('../config/insulin_config.yaml')\n",
    "with open(config_path, 'r') as f:\n",
    "    config = yaml.safe_load(f)\n",
    "\n",
    "print(f\"Review: {config['review_name']}\")\n",
    "print(f\"Inclusion terms: {len(config['inclusion_terms']['population'])} population terms\")\n",
    "print(f\"Exclusion terms: {len(config['exclusion_terms']['population'])} population terms\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Explore RIS Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load RIS file\n",
    "ris_path = Path('../data/insulin/input.ris')\n",
    "print(f\"Loading RIS file: {ris_path}\")\n",
    "\n",
    "try:\n",
    "    with open(ris_path, 'r', encoding='utf-8') as f:\n",
    "        ris_entries = rispy.load(f)\n",
    "except UnicodeDecodeError:\n",
    "    with open(ris_path, 'r', encoding='latin-1') as f:\n",
    "        ris_entries = rispy.load(f)\n",
    "\n",
    "print(f\"Loaded {len(ris_entries)} RIS entries\")\n",
    "\n",
    "# Convert to DataFrame\n",
    "ris_data = []\n",
    "for i, entry in enumerate(ris_entries):\n",
    "    record = {\n",
    "        'RIS_ID': f\"RIS_{i:04d}\",\n",
    "        'Title': entry.get('title', ''),\n",
    "        'Authors': '; '.join(entry.get('authors', [])) if entry.get('authors') else '',\n",
    "        'Abstract': entry.get('abstract', ''),\n",
    "        'Year': entry.get('year', ''),\n",
    "        'Journal': entry.get('journal_name', '') or entry.get('secondary_title', ''),\n",
    "        'DOI': entry.get('doi', ''),\n",
    "        'Accession_Number': entry.get('accession_number', ''),\n",
    "        'Type': entry.get('type', ''),\n",
    "        'Keywords': '; '.join(entry.get('keywords', [])) if entry.get('keywords') else '',\n",
    "        'URL': entry.get('url', ''),\n",
    "        'Abstract_Length': len(entry.get('abstract', '')),\n",
    "        'Has_Abstract': bool(entry.get('abstract', '')),\n",
    "        'Has_DOI': bool(entry.get('doi', '')),\n",
    "        'Has_Accession': bool(entry.get('accession_number', ''))\n",
    "    }\n",
    "    ris_data.append(record)\n",
    "\n",
    "ris_df = pd.DataFrame(ris_data)\n",
    "print(f\"\\nRIS DataFrame shape: {ris_df.shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display basic statistics\n",
    "print(\"📊 RIS Data Overview:\")\n",
    "print(f\"Total records: {len(ris_df)}\")\n",
    "print(f\"Records with abstract: {ris_df['Has_Abstract'].sum()} ({ris_df['Has_Abstract'].mean():.1%})\")\n",
    "print(f\"Records with DOI: {ris_df['Has_DOI'].sum()} ({ris_df['Has_DOI'].mean():.1%})\")\n",
    "print(f\"Records with Accession Number: {ris_df['Has_Accession'].sum()} ({ris_df['Has_Accession'].mean():.1%})\")\n",
    "\n",
    "# Year distribution\n",
    "print(f\"\\n📅 Publication Years:\")\n",
    "year_counts = ris_df['Year'].value_counts().sort_index()\n",
    "print(f\"Year range: {year_counts.index.min()} - {year_counts.index.max()}\")\n",
    "print(f\"Most common year: {year_counts.idxmax()} ({year_counts.max()} records)\")\n",
    "\n",
    "# Journal distribution\n",
    "print(f\"\\n📚 Top Journals:\")\n",
    "top_journals = ris_df['Journal'].value_counts().head(10)\n",
    "for journal, count in top_journals.items():\n",
    "    print(f\"  {journal}: {count} records\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize RIS data\n",
    "fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n",
    "\n",
    "# 1. Publication year distribution\n",
    "year_counts = ris_df['Year'].value_counts().sort_index()\n",
    "axes[0, 0].bar(year_counts.index.astype(str), year_counts.values)\n",
    "axes[0, 0].set_title('Publication Year Distribution', fontsize=14)\n",
    "axes[0, 0].set_xlabel('Year')\n",
    "axes[0, 0].set_ylabel('Number of Records')\n",
    "axes[0, 0].tick_params(axis='x', rotation=45)\n",
    "\n",
    "# 2. Abstract length distribution\n",
    "axes[0, 1].hist(ris_df['Abstract_Length'].dropna(), bins=50, edgecolor='black')\n",
    "axes[0, 1].set_title('Abstract Length Distribution', fontsize=14)\n",
    "axes[0, 1].set_xlabel('Abstract Length (characters)')\n",
    "axes[0, 1].set_ylabel('Frequency')\n",
    "axes[0, 1].axvline(ris_df['Abstract_Length'].median(), color='red', linestyle='--', \n",
    "                   label=f'Median: {ris_df[\"Abstract_Length\"].median():.0f}')\n",
    "axes[0, 1].legend()\n",
    "\n",
    "# 3. Identifier availability\n",
    "identifier_counts = [ris_df['Has_DOI'].sum(), ris_df['Has_Accession'].sum(), \n",
    "                     (ris_df['Has_DOI'] & ris_df['Has_Accession']).sum()]\n",
    "identifier_labels = ['Has DOI', 'Has Accession', 'Has Both']\n",
    "axes[1, 0].bar(identifier_labels, identifier_counts, color=['skyblue', 'lightgreen', 'salmon'])\n",
    "axes[1, 0].set_title('Identifier Availability', fontsize=14)\n",
    "axes[1, 0].set_ylabel('Number of Records')\n",
    "for i, v in enumerate(identifier_counts):\n",
    "    axes[1, 0].text(i, v, str(v), ha='center', va='bottom', fontweight='bold')\n",
    "\n",
    "# 4. Study types\n",
    "type_counts = ris_df['Type'].value_counts().head(10)\n",
    "axes[1, 1].barh(range(len(type_counts)), type_counts.values)\n",
    "axes[1, 1].set_yticks(range(len(type_counts)))\n",
    "axes[1, 1].set_yticklabels(type_counts.index)\n",
    "axes[1, 1].set_title('Top 10 Study Types', fontsize=14)\n",
    "axes[1, 1].set_xlabel('Number of Records')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Explore Covidence Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load Covidence data\n",
    "covidence_path = Path('../data/insulin/covidence_total.csv')\n",
    "print(f\"Loading Covidence data: {covidence_path}\")\n",
    "\n",
    "try:\n",
    "    covidence_df = pd.read_csv(covidence_path, encoding='utf-8')\n",
    "except UnicodeDecodeError:\n",
    "    covidence_df = pd.read_csv(covidence_path, encoding='latin-1')\n",
    "\n",
    "print(f\"Covidence DataFrame shape: {covidence_df.shape}\")\n",
    "print(f\"\\nColumns: {list(covidence_df.columns)}\")\n",
    "\n",
    "# Clean column names\n",
    "covidence_df.columns = [col.strip().lower().replace(' ', '_') for col in covidence_df.columns]\n",
    "print(f\"\\nCleaned columns: {list(covidence_df.columns)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Explore Covidence data\n",
    "print(\"📊 Covidence Data Overview:\")\n",
    "print(f\"Total records: {len(covidence_df)}\")\n",
    "\n",
    "# Check for decision column\n",
    "decision_col = None\n",
    "for col in covidence_df.columns:\n",
    "    if 'decision' in col.lower():\n",
    "        decision_col = col\n",
    "        break\n",
    "\n",
    "if decision_col:\n",
    "    print(f\"\\nDecision column found: '{decision_col}'\")\n",
    "    decision_counts = covidence_df[decision_col].value_counts(dropna=False)\n",
    "    for decision, count in decision_counts.items():\n",
    "        print(f\"  {decision}: {count} records ({count/len(covidence_df):.1%})\")\n",
    "else:\n",
    "    print(\"\\nNo decision column found in Covidence data\")\n",
    "    # Show first few rows\n",
    "    print(\"\\nFirst 5 rows:\")\n",
    "    print(covidence_df.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check identifier columns in Covidence\n",
    "print(\"🔍 Checking identifier columns in Covidence:\")\n",
    "for col in covidence_df.columns:\n",
    "    if any(term in col.lower() for term in ['doi', 'accession', 'id', 'number']):\n",
    "        non_null = covidence_df[col].notna().sum()\n",
    "        print(f\"  {col}: {non_null} non-null values ({non_null/len(covidence_df):.1%})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Text Analysis of Abstracts"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze text for common terms\n",
    "from collections import Counter\n",
    "import re\n",
    "\n",
    "def extract_common_terms(texts, n=20):\n",
    "    \"\"\"Extract most common terms from list of texts\"\"\"\n",
    "    all_words = []\n",
    "    for text in texts:\n",
    "        if isinstance(text, str):\n",
    "            # Simple word extraction\n",
    "            words = re.findall(r'\\b[a-zA-Z]{4,}\\b', text.lower())\n",
    "            all_words.extend(words)\n",
    "    \n",
    "    word_counts = Counter(all_words)\n",
    "    return word_counts.most_common(n)\n",
    "\n",
    "# Get abstracts\n",
    "abstracts = ris_df['Abstract'].dropna().tolist()\n",
    "common_terms = extract_common_terms(abstracts, 20)\n",
    "\n",
    "print(\"🔤 Most common terms in abstracts:\")\n",
    "for term, count in common_terms:\n",
    "    print(f\"  {term}: {count}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check for inclusion/exclusion terms\n",
    "print(\"🔍 Checking for inclusion terms in abstracts:\")\n",
    "\n",
    "# Flatten all inclusion terms\n",
    "all_inclusion_terms = []\n",
    "for category in ['population', 'intervention_and_comparator', 'study_design']:\n",
    "    all_inclusion_terms.extend(config['inclusion_terms'][category])\n",
    "\n",
    "term_counts = {}\n",
    "for term in all_inclusion_terms:\n",
    "    count = ris_df['Abstract'].str.contains(term, case=False, na=False).sum()\n",
    "    if count > 0:\n",
    "        term_counts[term] = count\n",
    "\n",
    "# Sort by frequency\n",
    "sorted_terms = sorted(term_counts.items(), key=lambda x: x[1], reverse=True)\n",
    "\n",
    "print(f\"Found {len(sorted_terms)} inclusion terms in abstracts:\")\n",
    "for term, count in sorted_terms[:15]:  # Show top 15\n",
    "    print(f\"  {term}: {count} records\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Data Quality Assessment"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Data quality metrics\n",
    "print(\"📈 DATA QUALITY ASSESSMENT\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "# RIS data quality\n",
    "ris_quality = {\n",
    "    'Total_Records': len(ris_df),\n",
    "    'With_Abstract': ris_df['Has_Abstract'].sum(),\n",
    "    'With_DOI': ris_df['Has_DOI'].sum(),\n",
    "    'With_Accession': ris_df['Has_Accession'].sum(),\n",
    "    'With_Year': ris_df['Year'].notna().sum(),\n",
    "    'With_Journal': ris_df['Journal'].notna().sum(),\n",
    "}\n",
    "\n",
    "print(\"\\nRIS Data Quality:\")\n",
    "for metric, count in ris_quality.items():\n",
    "    percentage = count / ris_quality['Total_Records'] * 100\n",
    "    print(f\"  {metric}: {count} ({percentage:.1f}%)\")\n",
    "\n",
    "# Covidence data quality\n",
    "if 'covidence_df' in locals():\n",
    "    print(\"\\nCovidence Data Quality:\")\n",
    "    print(f\"  Total records: {len(covidence_df)}\")\n",
    "    \n",
    "    # Find potential identifier columns\n",
    "    id_cols = []\n",
    "    for col in covidence_df.columns:\n",
    "        if any(term in col.lower() for term in ['doi', 'accession', 'id']):\n",
    "            id_cols.append(col)\n",
    "    \n",
    "    if id_cols:\n",
    "        print(f\"  Identifier columns found: {id_cols}\")\n",
    "        for col in id_cols:\n",
    "            non_null = covidence_df[col].notna().sum()\n",
    "            print(f\"    {col}: {non_null} non-null ({non_null/len(covidence_df)*100:.1f}%)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Save Analysis Report"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create summary report\n",
    "report = {\n",
    "    'review_name': config['review_name'],\n",
    "    'analysis_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),\n",
    "    'ris_data': {\n",
    "        'total_records': len(ris_df),\n",
    "        'years_range': f\"{ris_df['Year'].min()} - {ris_df['Year'].max()}\",\n",
    "        'with_abstract': int(ris_df['Has_Abstract'].sum()),\n",
    "        'with_abstract_pct': float(ris_df['Has_Abstract'].mean() * 100),\n",
    "        'with_doi': int(ris_df['Has_DOI'].sum()),\n",
    "        'with_doi_pct': float(ris_df['Has_DOI'].mean() * 100),\n",
    "        'abstract_length_median': float(ris_df['Abstract_Length'].median()),\n",
    "    },\n",
    "    'covidence_data': {\n",
    "        'total_records': len(covidence_df) if 'covidence_df' in locals() else 0,\n",
    "        'columns': list(covidence_df.columns) if 'covidence_df' in locals() else [],\n",
    "    },\n",
    "    'term_analysis': {\n",
    "        'inclusion_terms_found': len(sorted_terms),\n",
    "        'top_terms': dict(sorted_terms[:10]),\n",
    "    }\n",
    "}\n",
    "\n",
    "# Save report\n",
    "output_dir = Path('../data/output/exploration')\n",
    "output_dir.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "report_path = output_dir / 'data_exploration_report.json'\n",
    "with open(report_path, 'w') as f:\n",
    "    json.dump(report, f, indent=2)\n",
    "\n",
    "print(f\"✅ Analysis complete! Report saved to: {report_path}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "cochrane-screening",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}